In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import folium
from folium.plugins import AntPath
from folium import Map, PolyLine
from branca.element import Template, MacroElement
from branca.element import Element

def plot_routes_on_map(actual_coords, predicted_coords, metrics):
    # 지도 초기화
    m = folium.Map(location=actual_coords[0], zoom_start=8)

    # 실제 경로 (파란색 실선)
    folium.PolyLine(
        actual_coords, color='blue', weight=5, opacity=0.7, tooltip='Actual Route'
    ).add_to(m)

    # 예측 경로 (빨간색 점선)
    folium.PolyLine(
        predicted_coords, color='red', weight=3, opacity=0.7, tooltip='Predicted Route', dash_array='10'
    ).add_to(m)
     # 실제 경로 점 표시 (파란색)
    for lat, lon in actual_coords:
        folium.CircleMarker(
            location=(lat, lon),
            radius=4,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=f"Actual: ({lat:.5f}, {lon:.5f})"
        ).add_to(m)

    # 예측 경로 점 표시 (빨간색)
    for lat, lon in predicted_coords:
        folium.CircleMarker(
            location=(lat, lon),
            radius=4,
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.7,
            popup=f"Predicted: ({lat:.5f}, {lon:.5f})"
        ).add_to(m)
    # 범례 + 평가 지표 HTML
    legend_html = f"""
    <div style="
        position: fixed;
        bottom: 50px;
        left: 50px;
        z-index: 9999;
        background-color: white;
        border: 2px solid grey;
        border-radius: 5px;
        padding: 10px;
        font-size: 14px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
        line-height: 1.6;
    ">
      <b>🗺 경로 범례</b><br>
      <i style="background:blue; width:10px; height:10px; display:inline-block;"></i> 실제 경로<br>
      <i style="width:20px; height:0px; border-top: 3px dashed red; display:inline-block;"></i> 예측 경로
      <hr style="margin:6px 0;">
      <b>평가 지표</b><br>
      Haversine 평균 거리: {metrics['haversine_km']:.3f} km<br>
      유사도: {metrics['similarity_percent']:.2f} %<br>
      RMSE: {metrics['rmse']:.5f}<br>
      MSE: {metrics['mse']:.5f}
    </div>
    """

    # HTML 요소를 지도에 추가
    m.get_root().html.add_child(Element(legend_html))

    return m


temp_df = df.iloc[12:]
values = temp_df[['위도', '경도']].values

actual_coords = values
predicted_coords = predictions[:,:2]

from haversine import haversine

def haversine_average_distance(actual, predicted):
    min_len = min(len(actual), len(predicted))
    total_distance = sum(haversine(a, p) for a, p in zip(actual[:min_len], predicted[:min_len]))
    return total_distance / min_len  # km


import numpy as np
from sklearn.metrics import mean_squared_error

def coord_rmse_mse(actual, predicted):
    min_len = min(len(actual), len(predicted))
    actual_np = np.array(actual[:min_len])
    pred_np = np.array(predicted[:min_len])

    mse = mean_squared_error(actual_np, pred_np)
    rmse = np.sqrt(mse)
    return rmse, mse


def evaluate_route(actual_coords, predicted_coords):
    # Haversine 평균 거리
    haversine_avg = haversine_average_distance(actual_coords, predicted_coords)

    # Haversine 기반 퍼센트 유사도 (100km 이상은 0%)
    similarity = max(0, 100 * (1 - haversine_avg / 100))

    # 위도/경도 기준 RMSE & MSE
    rmse, mse = coord_rmse_mse(actual_coords, predicted_coords)

    print(f"📍 평균 거리 (Haversine): {haversine_avg:.3f} km")
    print(f"✅ 유사도: {similarity:.2f}%")
    print(f"📉 RMSE (위도+경도 좌표): {rmse:.6f}")
    print(f"📉 MSE  (위도+경도 좌표): {mse:.6f}")

    return {
        'haversine_km': haversine_avg,
        'similarity_percent': similarity,
        'rmse': rmse,
        'mse': mse,
    }

    
metrics = evaluate_route(actual_coords, predicted_coords)
m = plot_routes_on_map(actual_coords, predicted_coords, metrics)
m.save('ph_test.html')
print(len(actual_coords), len(predicted_coords)) #8